# Go2 ODD/COD Observer - Streamlined Workflow

**Run the complete 10-agent ODD/COD analysis pipeline**

This notebook executes the tested `odd_workflow_full.py` script with visualizations.

## What it does:
1. ODD Spec → Perception → Motion → Collision → COD → Compliance → Report
2. IMU-based motion detection (robust to odometry failures)
3. Sim vs real classification
4. Comprehensive compliance analysis

**Runtime:** 2-3 minutes for 13-window scenario

## Setup

In [ ]:
import os
import sys
import json
import asyncio
from pathlib import Path

# Add scripts to path
sys.path.insert(0, str(Path().absolute().parent / 'scripts'))

# Check API key
from dotenv import load_dotenv
load_dotenv()

if not os.getenv('GOOGLE_API_KEY'):
    print('❌ Set GOOGLE_API_KEY environment variable')
    print('Get key: https://aistudio.google.com/app/apikey')
else:
    print('✅ API key configured')

# Visualization
import matplotlib.pyplot as plt
import pandas as pd
print('✅ Setup complete')

## Configure Analysis

In [ ]:
# Select scenario
SCENARIO = 'sim_run_test'  # Change to your scenario

# Custom ODD (optional - uses default if None)
CUSTOM_ODD = None

print(f'Scenario: {SCENARIO}')
print(f'ODD: {"Default (indoor office)" if not CUSTOM_ODD else "Custom"}')

## Run Workflow

In [ ]:
from odd_workflow_full import run_odd_workflow

print('🚀 Starting 10-agent workflow...')
print('This takes 2-3 minutes. Please wait...\n')

result = await run_odd_workflow(
    scenario_name=SCENARIO,
    nl_odd_description=CUSTOM_ODD
)

print('\n✅ Workflow complete!')
print(f'Windows: {result["report"]["scenario_metadata"]["total_windows_analyzed"]}')
print(f'Source: {result["report"]["scenario_metadata"]["data_source"]}')
print(f'Compliance: {result["full_analysis"]["odd_compliance"]["overall_compliance"]}')

## View Results

In [ ]:
print('='*80)
print('EXECUTIVE SUMMARY')
print('='*80)
print(result['report']['executive_summary'])
print('\n' + '='*80)
print('KEY FINDINGS')
print('='*80)
for i, finding in enumerate(result['report']['key_findings'], 1):
    print(f'{i}. {finding}')

## Collision Risk Visualization

In [ ]:
collision = result['full_analysis']['collision']
windows = [w['window_id'] for w in collision['per_window_collision']]
risks = [w['likelihood'] for w in collision['per_window_collision']]

plt.figure(figsize=(12, 5))
plt.plot(risks, 'o-', linewidth=2, markersize=8)
plt.axhline(0.3, color='orange', linestyle='--', label='Boundary')
plt.axhline(0.5, color='red', linestyle='--', label='Alert')
plt.fill_between(range(len(risks)), 0, 0.3, alpha=0.1, color='green')
plt.fill_between(range(len(risks)), 0.3, 0.5, alpha=0.1, color='orange')
plt.fill_between(range(len(risks)), 0.5, 1.0, alpha=0.1, color='red')
plt.xlabel('Window')
plt.ylabel('Collision Likelihood')
plt.title('Collision Risk Timeline')
plt.xticks(range(len(windows)), windows)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Full Analysis JSON

In [ ]:
# Display formatted JSON
print(json.dumps(result, indent=2))